# Data quality validation

Check table availability, null resource IDs, duplicate logical IDs and uniqueness of active SCD2 versions. Persist results and fail on critical checks.

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
resources = ["patient","encounter","observation","condition"]
results = []
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

for r in resources:
    table = f"silver_{r}"
    if not spark.catalog.tableExists(table):
        results.append((run_id,table,"TABLE_EXISTS",0,0,"FAIL","Silver table missing"))
        continue
    df = spark.table(table)
    total = df.count()
    null_ids = df.filter(F.col("resource_id").isNull()).count()
    duplicate_ids = df.groupBy("resource_id").count().filter(F.col("count") > 1).count()
    results.append((run_id,table,"ROW_COUNT",total,0,"PASS",f"rows={total}"))
    results.append((run_id,table,"NULL_RESOURCE_ID",null_ids,0,"PASS" if null_ids == 0 else "FAIL",f"null_ids={null_ids}"))
    results.append((run_id,table,"DUPLICATE_RESOURCE_ID",duplicate_ids,0,"PASS" if duplicate_ids == 0 else "FAIL",f"duplicate_ids={duplicate_ids}"))

    history = f"history_{r}"
    if spark.catalog.tableExists(history):
        active_duplicates = (spark.table(history).filter(F.col("is_active") == True)
            .groupBy("resource_id").count().filter(F.col("count") > 1).count())
        results.append((run_id,history,"ONE_ACTIVE_VERSION",active_duplicates,0,
                        "PASS" if active_duplicates == 0 else "FAIL",
                        f"resource_ids_with_multiple_active_versions={active_duplicates}"))

schema = "run_id string, table_name string, check_name string, observed_value long, expected_value long, status string, details string"
quality_df = spark.createDataFrame(results, schema)
if spark.catalog.tableExists("metadata_data_quality_results"):
    quality_df.write.format("delta").mode("append").saveAsTable("metadata_data_quality_results")
else:
    quality_df.write.format("delta").mode("overwrite").saveAsTable("metadata_data_quality_results")

display(quality_df)
failed = quality_df.filter(F.col("status") == "FAIL").count()
if failed:
    raise RuntimeError(f"Data quality failed: {failed} checks. See metadata_data_quality_results.")
